In [1]:
import torch
import torch.nn as nn
from torch import Tensor
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader,SubsetRandomSampler,ConcatDataset
import torch.nn.functional as F

import boda

In [2]:
from akita_coda_helper import plot_map, from_upper_triu

In [3]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

from model import SeqNN

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(device)

cuda:0


In [5]:
# Load the entire model (architecture + weights)
model = torch.load("/home1/smaruj/pytorch_akita/model.pth")

/tmp/SLURM_209730/ipykernel_2046228/3921338739.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("/home1/smaruj/pytorch_akita/model.pth")


In [6]:
model = model.to(device)

In [7]:
# Set the model to evaluation mode (important for inference)
model.eval()

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (re_lu): ReLU()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 96, kernel_size=(11,), stride=(1,), padding=(5,), bias=False)
    (batch_norm): BatchNorm1d(96, eps=0.001, momentum=0.9265, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(96, 96, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(96, eps=0.001, momentum=0.9265, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(96, 96, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(96, eps=0.001, momentum=0.9265, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size

In [8]:
from torchinfo import summary

summary(model, input_size=(2, 4, 1048576), col_names=["output_size", "num_params"])

Layer (type:depth-idx)                   Output Shape              Param #
SeqNN                                    [2, 5, 99681]             --
├─StochasticReverseComplement: 1-1       [2, 4, 1048576]           --
├─StochasticShift: 1-2                   [2, 4, 1048576]           --
├─ReLU: 1-3                              [2, 4, 1048576]           --
├─ConvBlock: 1-4                         [2, 96, 524288]           --
│    └─Conv1d: 2-1                       [2, 96, 1048576]          4,224
│    └─BatchNorm1d: 2-2                  [2, 96, 1048576]          192
│    └─MaxPool1d: 2-3                    [2, 96, 524288]           --
├─ConvTower: 1-5                         [2, 96, 512]              --
│    └─Sequential: 2-4                   [2, 96, 512]              --
│    │    └─ReLU: 3-1                    [2, 96, 524288]           --
│    │    └─Conv1d: 3-2                  [2, 96, 524288]           46,080
│    │    └─BatchNorm1d: 3-3             [2, 96, 524288]           192
│    │

In [9]:
from pyfaidx import Fasta

In [10]:
fasta_file = "/project/fudenber_735/genomes/hg38/hg38.fa"
genome = Fasta(fasta_file)

region = "chr12"
start = 115163136
end = 116211712

# region = "chr11"
# start = 75429888
# end = 76478464

# region = "chr15"
# start = 63281152
# end = 64329728

In [11]:
sequence = genome[region][start:end]

In [12]:
genome.close()

In [13]:
import numpy as np

In [14]:
def one_hot_encode_sequence(sequence_obj):
    # Convert pyfaidx.Sequence object to string
    sequence = str(sequence_obj).upper()

    # Define the mapping from bases to integers
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

    # Step 1: Encode the sequence to integers
    encoded_sequence = np.array([base_to_int[base] for base in sequence])

    # Step 2: One-hot encode the sequence
    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    # Step 3: Expand the dimensions to [1, 4, sequence_length]
    input_sequence = np.expand_dims(one_hot_encoded, axis=0)

    return input_sequence

In [15]:
X = one_hot_encode_sequence(sequence)

del sequence

In [16]:
# Convert the NumPy array to a PyTorch tensor
X_torch = torch.tensor(X)

# Now you can move the tensor to the desired device (e.g., GPU or CPU)
# X = X_torch.to(device)
X = X_torch.to("cuda:0")

In [17]:
# Function to get the indices of the upper triangular part for a given matrix size
def get_upper_triu_indices(dim, num_diags=2):
    return np.triu_indices(dim, k=num_diags)

# The matrix size you are working with
dim = 64
# The starting and ending indices for a chunk
start_row, end_row = 112, 256
start_col, end_col = 256, 368

# Get the full upper triangular indices for the 64x64 matrix
full_indices = get_upper_triu_indices(dim=448)

# Now create a mask to extract the relevant slice of the matrix
mask = ((full_indices[0] >= start_row) & (full_indices[0] < end_row) &
        (full_indices[1] >= start_col) & (full_indices[1] < end_col))

# Extract the corresponding indices from the full 64x64 vector
sub_indices_in_full_vector = np.where(mask)[0]



In [18]:
sub_indices_in_full_vector

array([43878, 43879, 43880, ..., 81453, 81454, 81455])

In [19]:
def indices_to_slices_fixed(indices):
    slices = []
    start = indices[0]
    
    for i in range(1, len(indices)):
        if indices[i] != indices[i-1] + 1:  # New contiguous block detected
            slices.append(slice(start, indices[i-1] + 1))
            start = indices[i]
    
    # Append the last slice
    slices.append(slice(start, indices[-1] + 1))
    
    # Compute the max slice length
    max_length = max(s.stop - s.start for s in slices)

    # Expand shorter slices to match max_length
    slices_fixed = []
    for s in slices:
        length = s.stop - s.start
        if length < max_length:
            new_stop = s.start + max_length  # Expand
            slices_fixed.append(slice(s.start, new_stop))
        else:
            slices_fixed.append(s)
    
    return slices_fixed

In [20]:
slices = indices_to_slices_fixed(sub_indices_in_full_vector)

In [21]:
slices

[slice(43878, 43990, None),
 slice(44211, 44323, None),
 slice(44543, 44655, None),
 slice(44874, 44986, None),
 slice(45204, 45316, None),
 slice(45533, 45645, None),
 slice(45861, 45973, None),
 slice(46188, 46300, None),
 slice(46514, 46626, None),
 slice(46839, 46951, None),
 slice(47163, 47275, None),
 slice(47486, 47598, None),
 slice(47808, 47920, None),
 slice(48129, 48241, None),
 slice(48449, 48561, None),
 slice(48768, 48880, None),
 slice(49086, 49198, None),
 slice(49403, 49515, None),
 slice(49719, 49831, None),
 slice(50034, 50146, None),
 slice(50348, 50460, None),
 slice(50661, 50773, None),
 slice(50973, 51085, None),
 slice(51284, 51396, None),
 slice(51594, 51706, None),
 slice(51903, 52015, None),
 slice(52211, 52323, None),
 slice(52518, 52630, None),
 slice(52824, 52936, None),
 slice(53129, 53241, None),
 slice(53433, 53545, None),
 slice(53736, 53848, None),
 slice(54038, 54150, None),
 slice(54339, 54451, None),
 slice(54639, 54751, None),
 slice(54938, 55050,

# CODA

In [ ]:
# Fast SeqProp

In [ ]:
(1048576 / 2) + 200

In [ ]:
X.shape

In [22]:
logits = torch.randn(
    1, 4, 400
)

left_flank = X[:,:, :524088] #.unsqueeze(0)

right_flank = X[:,:, :-524488] #.unsqueeze(0)

In [23]:
left_flank.shape, right_flank.shape

(torch.Size([1, 4, 524088]), torch.Size([1, 4, 524088]))

In [ ]:
524088 * 2 + 400

In [24]:
params = boda.generator.StraightThroughParameters(
    data=logits, left_flank=left_flank, right_flank=right_flank, 
    n_samples=1, use_affine=False
)
params.cuda()

StraightThroughParameters(
  (norm): InstanceNorm1d(4, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
)

In [25]:
# energy = boda.generator.MinGapEnergy(model, target_feature=21, target_alpha=-1, a_min=-2., a_max=6.)
# energy = boda.generator.MinGapEnergy(model, target_feature=slice(96,980), target_alpha=-1, a_min=-2., a_max=6.)
energy = boda.generator.MinGapEnergy(model, target_feature=slices, target_alpha=1, a_min=-2., a_max=6.)

#Higher or Lower?:
# - If you want the target feature to be higher in the energy calculation (i.e., the energy should be lower when the target feature is higher), 
# then you’ll use a positive target_alpha.
# - If you want the target feature to be lower (i.e., the energy should be lower when the target feature is lower), 
# then you’ll use a negative target_alpha.

In [26]:
FSP = boda.generator.FastSeqProp(energy, params)

In [27]:
FSP.run(n_steps=400, lr_scheduler=True, create_plot=True)
proposals = params.get_sample()
proposals.shape

Steps: 100%|██████████| 400/400 [18:26<00:00,  2.77s/it, Loss=4.18e+3, LR=1e-6]    


(400,)
(2000,)
(1, 5)


ValueError: All arrays must be of the same length

In [ ]:
squeezed_proposals = proposals.squeeze(0)

In [ ]:
squeezed_proposals.shape, logits.shape

In [ ]:
og_seq = X_torch

In [ ]:
og_seq.shape

In [ ]:
og_seq[:, 10328:10664] = squeezed_proposals

In [ ]:
og_seq_dim = og_seq.unsqueeze(0)

In [ ]:
with torch.no_grad():
    predictions_new = model(og_seq_dim)

In [ ]:
predictions_new.shape

In [ ]:
plot_map(from_upper_triu(predictions[1], matrix_len=64, num_diags=2), vmin=-0.6, vmax=0.6, palette="RdBu_r", width=5, height=5)

In [ ]:
plot_map(from_upper_triu(predictions_new, matrix_len=64, num_diags=2), vmin=-0.6, vmax=0.6, palette="RdBu_r", width=5, height=5)

In [ ]:
map_one = from_upper_triu(predictions[1], matrix_len=64, num_diags=2)
map_two = from_upper_triu(predictions_new, matrix_len=64, num_diags=2)

In [ ]:
np.nanmean(map_one), np.nanmean(map_two)